In [13]:
from typing import List, Dict, Tuple, Any
from dataclasses import dataclass
import numpy as np
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownTextSplitter,
    PythonCodeTextSplitter,
    HTMLHeaderTextSplitter
)
from langchain_openai import OpenAIEmbeddings

Using datatclass over dictionary is to help with
* enforcing consistent schema(data type like avg_chunk_float: float and it accepts only float values)
* improve structure and readability
* helpful in multiple metrics and helps with aggregating results in padas for analysis

In [14]:
@dataclass
class ChunkMetrics:
    splitter_name: str
    avg_chunk_size: float
    num_chunks: int
    size_std_dev:float
    overlap_ratio: float
    precision: float = 0.0
    recall: float = 0.0
    iou: float = 0.0

In [33]:
class ChunkingEvaluator:

    def __init__(self, sample_texts: Dict[str, str]):
        # sample_texts: Dict with keys like 'general', 'markdown', 'code', 'html'
        self.sample_texts = sample_texts
        self.results = []

    """ This is for creating a basic character splitter """
    def get_character_splitter(self, chunk_size: int=1000, chunk_overlap: int=200):
        return CharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separator="\n\n"
        )
    
    """ This is for creating a recursive character splitter """
    def get_recursive_character_splitter(self, chunk_size: int=1000, chunk_overlap: int=200):
        return RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )
    
    """ This is for creating a markdown splitter """
    def get_markdown_splitter(self, chunk_size: int=1000, chunk_overlap=200):
        return MarkdownTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap
        )
    
    """ This is for creating a python code splitter """
    def get_python_code_splitter(self, chunk_size: int=1000, chunk_overlap: int=200):
        return PythonCodeTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap
        )
    
    """ This is for creating an HTML header splitter """
    def get_html_header_splitter(self):
        headers_to_split_on = [
            ("h1", "Header 1"),
            ("h2", "Header 2"),
            ("h3", "Header 3")
        ]
        return HTMLHeaderTextSplitter(
            headers_to_split_on = headers_to_split_on
        )
    
    """ This is for calculating overlap ratio between consecutive chunks """
    def calculate_overlap_ratio(self, chunks: List[str]):
        # If fewer than two chunks exist, overlap cannot be computed
        if len(chunks) < 2:
            return 0.0
        
        overlaps = []

        # Iterate over consecutive chunk pairs
        for i in range(len(chunks)-1):
            chunk1, chunk2 = chunks[i], chunks[i+1]
            # Compute the longest common substring between the two chunks eg: chunk1 = "Hello world", chunk2 = "world of AI" => overlap = "world"
            overlap = self._longest_common_substring(chunk1, chunk2)
            # overlap = len(set(chunk1.split()).intersection(set(chunk2.split())))

            # Avoid division by zero
            if len(chunk1) > 0:
                # Normalize overlap length by the size of the first chunk
                overlaps.append(len(overlap) / len(chunk1))

        return np.mean(overlaps) if overlaps else 0.0
    
    """ This is for finding the longest common substring between two strings """
    def _longest_common_substring(self, str1: str, str2: str) -> str:
        m, n = len(str1), len(str2)
        # DP table where dp[i][j] stores the length of the longest, common suffix of str1[0..i-1] and str2[0..j-1]
        dp = [[0]*(n+1) for _ in range(m+1)]
        # Track the maximum substring length found and its ending position in str1
        max_len = 0
        end_pos = 0

        for i in range(1, m+1):
            for j in range(1, n+1):
                # If characters match, extend the length of the common substring
                if str1[i-1] == str2[j-1]:
                    dp[i][j] = dp[i-1][j-1]+1
                    if dp[i][j] > max_len:
                        max_len = dp[i][j]
                        end_pos = i
        # Extract the longest common substring from str1 -> "world"
        return str1[end_pos-max_len:end_pos]

    def create_ground_truth_boundaries(self, text: str, text_type: str = 'general') -> List[int]:
        """
        Create ground truth boundaries based on natural splitting points in the text
        For general text: paragraph breaks
        For code: function/class boundaries
        For markdown: header boundaries
        For HTML: tag boundaries
        """
        boundaries = []
        
        if text_type == 'general':
            # Split on double newlines (paragraph breaks)
            pos = 0
            for paragraph in text.split('\n\n'):
                pos += len(paragraph) + 2  # +2 for '\n\n'
                if pos < len(text):
                    boundaries.append(pos)
        
        elif text_type == 'code':
            # Split on function/class definitions
            lines = text.split('\n')
            pos = 0
            for i, line in enumerate(lines):
                if line.strip().startswith(('def ', 'class ')):
                    if pos > 0:  # Don't add boundary at start
                        boundaries.append(pos)
                pos += len(line) + 1  # +1 for '\n'
        
        elif text_type == 'markdown':
            # Split on headers
            lines = text.split('\n')
            pos = 0
            for line in lines:
                if line.strip().startswith('#'):
                    if pos > 0:
                        boundaries.append(pos)
                pos += len(line) + 1
        
        elif text_type == 'html':
            # Split on header tags
            import re
            for match in re.finditer(r'<h[1-6]>', text):
                if match.start() > 0:
                    boundaries.append(match.start())
        
        return boundaries
     
    """ For Precision, Recall, IoU calculations """
    def calculate_precision_recall_iou(
        self, 
        chunks: List[str], 
        ground_truth_boundaries: List[int],
        debug: bool = False
    ) -> Tuple[float, float, float]:
        """
        Calculate precision, recall, and IOU for chunk boundaries
        
        Args:
            chunks: List of text chunks
            ground_truth_boundaries: List of character positions where chunks should split
            debug: If True, print debugging information
        
        Returns:
            Tuple of (precision, recall, iou)
        """
        # Calculate predicted boundaries
        predicted_boundaries = []
        pos = 0
        for chunk in chunks[:-1]:  # Exclude last chunk
            pos += len(chunk)
            predicted_boundaries.append(pos)
        
        if debug:
            print(f"\n  [DEBUG] Predicted boundaries: {predicted_boundaries}")
            print(f"  [DEBUG] Ground truth boundaries: {ground_truth_boundaries}")
        
        # If no boundaries to compare, return zeros
        if not predicted_boundaries or not ground_truth_boundaries:
            return 0.0, 0.0, 0.0
        
        # Convert to sets for comparison (with tolerance)
        tolerance = 100  # Allow 100 character tolerance
        
        true_positives = 0
        matched_gt = set()
        
        for pred in predicted_boundaries:
            for i, gt in enumerate(ground_truth_boundaries):
                if i not in matched_gt and abs(pred - gt) <= tolerance:
                    true_positives += 1
                    matched_gt.add(i)
                    if debug:
                        print(f"  [DEBUG] Match found: predicted={pred}, ground_truth={gt}, diff={abs(pred-gt)}")
                    break
        
        precision = true_positives / len(predicted_boundaries) if predicted_boundaries else 0
        recall = true_positives / len(ground_truth_boundaries) if ground_truth_boundaries else 0
        
        # IOU calculation
        if precision + recall > 0:
            iou = true_positives / (len(predicted_boundaries) + len(ground_truth_boundaries) - true_positives)
        else:
            iou = 0.0
        
        if debug:
            print(f"  [DEBUG] True positives: {true_positives}")
            print(f"  [DEBUG] Precision: {precision:.2%}, Recall: {recall:.2%}, IoU: {iou:.2%}")
        
        return precision, recall, iou
    
    """ This is for evaluating all splitters """
    def evaluate_splitter(self, splitter, text: str, splitter_name: str, ground_truth_boundaries: List[int] = None, debug: bool = False) -> ChunkMetrics:
        if isinstance(splitter, HTMLHeaderTextSplitter):
            chunks = splitter.split_text(text)
            chunks = [chunks.page_content for chunks in chunks]
        else:
            chunks = splitter.split_text(text)

        # Calculating the basic metrics
        chunk_sizes = [len(chunk) for chunk in chunks]
        avg_chunk_size = np.mean(chunk_sizes)
        size_std_dev = np.std(chunk_sizes)
        overlap_ratio = self.calculate_overlap_ratio(chunks)

        precision, recall, iou = 0.0, 0.0, 0.0
        if ground_truth_boundaries:
            precision, recall, iou = self.calculate_precision_recall_iou(chunks, ground_truth_boundaries)
        
        return ChunkMetrics(
            splitter_name=splitter_name,
            avg_chunk_size=avg_chunk_size,
            num_chunks=len(chunks),
            size_std_dev=size_std_dev,
            overlap_ratio=overlap_ratio,
            precision=precision,
            recall=recall,
            iou=iou
        )
    
    """ This is for running the full evaluation """
    def run_evaluation(
        self, 
        chunk_size: int = 1000, 
        overlap: int = 200,
        use_ground_truth: bool = True,
        debug: bool = False
    ):
        """Run evaluation on all splitters and text types"""
        
        print("=" * 80)
        print("CHUNKING STRATEGIES EVALUATION")
        print("=" * 80)
        
        # Test general text
        if 'general' in self.sample_texts:
            text = self.sample_texts['general']
            print(f"\n📄 Evaluating GENERAL TEXT ({len(text)} chars)")
            print("-" * 80)
            
            ground_truth = self.create_ground_truth_boundaries(text, 'general') if use_ground_truth else None
            if ground_truth:
                print(f"Ground truth boundaries: {len(ground_truth)} natural split points")
            
            splitters = [
                (self.get_character_splitter(chunk_size, overlap), "CharacterTextSplitter"),
                (self.get_recursive_character_splitter(chunk_size, overlap), "RecursiveCharacterTextSplitter"),
            ]
            
            for splitter, name in splitters:
                metrics = self.evaluate_splitter(splitter, text, name, ground_truth, debug=debug)
                self.results.append(metrics)
                self._print_metrics(metrics)
        
        # Test markdown
        if 'markdown' in self.sample_texts:
            text = self.sample_texts['markdown']
            print(f"\n📝 Evaluating MARKDOWN TEXT ({len(text)} chars)")
            print("-" * 80)
            
            ground_truth = self.create_ground_truth_boundaries(text, 'markdown') if use_ground_truth else None
            if ground_truth:
                print(f"Ground truth boundaries: {len(ground_truth)} natural split points")
            
            splitter = self.get_markdown_splitter(chunk_size, overlap)
            metrics = self.evaluate_splitter(splitter, text, "MarkdownTextSplitter", ground_truth, debug=debug)
            self.results.append(metrics)
            self._print_metrics(metrics)
        
        # Test Python code
        if 'code' in self.sample_texts:
            text = self.sample_texts['code']
            print(f"\n🐍 Evaluating PYTHON CODE ({len(text)} chars)")
            print("-" * 80)
            
            ground_truth = self.create_ground_truth_boundaries(text, 'code') if use_ground_truth else None
            if ground_truth:
                print(f"Ground truth boundaries: {len(ground_truth)} natural split points")
            
            splitter = self.get_python_code_splitter(chunk_size, overlap)
            metrics = self.evaluate_splitter(splitter, text, "PythonCodeTextSplitter", ground_truth, debug=debug)
            self.results.append(metrics)
            self._print_metrics(metrics)
        
        # Test HTML
        if 'html' in self.sample_texts:
            text = self.sample_texts['html']
            print(f"\n🌐 Evaluating HTML ({len(text)} chars)")
            print("-" * 80)
            
            ground_truth = self.create_ground_truth_boundaries(text, 'html') if use_ground_truth else None
            if ground_truth:
                print(f"Ground truth boundaries: {len(ground_truth)} natural split points")
            
            splitter = self.get_html_header_splitter()
            metrics = self.evaluate_splitter(splitter, text, "HTMLHeaderTextSplitter", ground_truth, debug=debug)
            self.results.append(metrics)
            self._print_metrics(metrics)
        
        self._print_summary()       


    """ This is for printing the metrics """
    def _print_metrics(self, metrics: ChunkMetrics):
        print(f"\n{metrics.splitter_name}: ")
        print(f"  • Number of chunks: {metrics.num_chunks}")
        print(f"  • Average chunk size: {metrics.avg_chunk_size:.2f} chars")
        print(f"  • Std deviation: {metrics.size_std_dev:.2f}")
        print(f"  • Overlap ratio: {metrics.overlap_ratio:.2%}")
        if metrics.precision > 0.0 or metrics.recall > 0.0:
            print(f"  • Precision: {metrics.precision:.2%}")
            print(f"  • Recall: {metrics.recall:.2%}")
            print(f"  • IOU: {metrics.iou:.2%}")

    """ This is for printing the summary of all results """
    def _print_summary(self):
        print("\n" + "="*80)
        print('Chunking Evaluation Summary')
        print("="*80)
        print(f"\n{'Splitter':<35} {'Chunks' :<10} {'Avg Size':<12} {'Overlap':<10}")
        print("-" * 80)
        for result in self.results:
            print(f"{result.splitter_name:<35}"
                  f"{result.num_chunks:< 10}"
                  f"{result.avg_chunk_size:<12.0f}"
                  f"{result.overlap_ratio:<10.2%}")

In [34]:
if __name__=="__main__":
    sample_texts = {
        'general': """Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.

There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.""",
'markdown': """
# Introduction to Machine Learning

## What is Machine Learning?

Machine learning is a subset of artificial intelligence that focuses on algorithms.

### Types of Machine Learning

1. **Supervised Learning**: Learning from labeled data
2. **Unsupervised Learning**: Finding patterns in unlabeled data
3. **Reinforcement Learning**: Learning through trial and error

## Applications

Machine learning is used in:
- Image recognition
- Natural language processing
- Recommendation systems
        """,
        'code': """
def train_model(data, labels, epochs=10):
    '''Train a machine learning model'''
    model = Model()
    
    for epoch in range(epochs):
        # Forward pass
        predictions = model.forward(data)
        
        # Calculate loss
        loss = calculate_loss(predictions, labels)
        
        # Backward pass
        model.backward(loss)
        
        # Update weights
        model.update_weights()
    
    return model

class Model:
    def __init__(self):
        self.weights = initialize_weights()
    
    def forward(self, x):
        return x @ self.weights
    
    def backward(self, loss):
        self.gradients = compute_gradients(loss)
    """,
        'html': """
<html>
<body>
<h1>Machine Learning Guide</h1>
<p>Introduction to ML concepts</p>

<h2>Supervised Learning</h2>
<p>Learn from labeled examples</p>

<h2>Unsupervised Learning</h2>
<p>Find patterns in data</p>

<h3>Clustering</h3>
<p>Group similar items together</p>
</body>
</html>
        """
    }
    evaluator = ChunkingEvaluator(sample_texts)
    evaluator.run_evaluation(
    chunk_size=500,
    overlap=100,
    use_ground_truth=True,
    debug=True  # This will show you what's happening
)

    print("\n✅ Evaluation complete!")


CHUNKING STRATEGIES EVALUATION

📄 Evaluating GENERAL TEXT (1334 chars)
--------------------------------------------------------------------------------
Ground truth boundaries: 2 natural split points

CharacterTextSplitter: 
  • Number of chunks: 3
  • Average chunk size: 443.33 chars
  • Std deviation: 209.14
  • Overlap ratio: 3.06%
  • Precision: 100.00%
  • Recall: 100.00%
  • IOU: 100.00%

RecursiveCharacterTextSplitter: 
  • Number of chunks: 4
  • Average chunk size: 350.00 chars
  • Std deviation: 83.65
  • Overlap ratio: 6.53%
  • Precision: 66.67%
  • Recall: 100.00%
  • IOU: 66.67%

📝 Evaluating MARKDOWN TEXT (497 chars)
--------------------------------------------------------------------------------
Ground truth boundaries: 4 natural split points

MarkdownTextSplitter: 
  • Number of chunks: 1
  • Average chunk size: 487.00 chars
  • Std deviation: 0.00
  • Overlap ratio: 0.00%

🐍 Evaluating PYTHON CODE (633 chars)
-----------------------------------------------------------